<a href="https://colab.research.google.com/github/eco-bios/monitoreo-deforestacion-chiapas/blob/main/Pipeline_Deforestacion_Chiapas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este Script permite autenticar la cuenta e inicializa la conexion con el proyecto y se verifica mediante un dataset la conexion a los servidores de google earth

In [ ]:
import ee
import warnings
# Esto ignora los mensajes de advertencia que no son críticos
warnings.filterwarnings('ignore')

# --- AUTENTICACIÓN ---
# Inicia el proceso de autorización para acceder a la cuenta de Earth Engine.
ee.Authenticate()

try:
    # --- INICIALIZACIÓN ---
    # Intenta establecer conexión con el proyecto específico en Google Cloud
    ee.Initialize(project='monitoreo-deforestacion-ch')

    # --- PRUEBA DE CONEXIÓN ---
    # Se carga un dataset global (SRTM) para verificar el acceso a los servidores
    test_image = ee.Image("USGS/SRTMGL1_003")
    print("✅ ¡CONECTADO! El motor de Google Earth Engine ya te reconoce.")

except Exception as e:
    # Mensaje en caso de fallo en las credenciales o el ID del proyecto
    print(f"❌ Error detallado: {e}")

Este script realiza el procesamiento de imágenes satelitales para calcular el vigor vegetativo en la zona de Escuintla, Chiapas. El flujo de trabajo optimiza la selección de datos y reduce la interferencia climática (nubes) común en regiones de sierra.

In [ ]:
import ee

# 1. Definición del sitio de interés: San Juan Panamá, Escuintla, Chiapas
# Se utiliza una geometría de punto para la consulta espacial
punto_escuintla = ee.Geometry.Point([-92.60, 15.29])

# 2. Filtrado de Colección Sentinel-2 (Nivel 2A - Reflectancia en Superficie)
# Filtramos por ubicación, rango temporal y un umbral de nubosidad del 20%
# Nota: Se permite un margen mayor de nubes debido a las condiciones climáticas de la zona de sierra.
coleccion_escuintla = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(punto_escuintla)
    .filterDate('2024-01-01', '2024-01-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

# 3. Composición del mosaico mediante Reductor de Mediana
# Esto ayuda a eliminar valores atípicos y restos de nubes/sombras
mosaico_escuintla = coleccion_escuintla.median()

# 4. Cálculo del Índice de Vegetación de Diferencia Normalizada (NDVI)
# Fórmula: (NIR - Red) / (NIR + Red) -> Bandas B8 y B4 en Sentinel-2
ndvi_escuintla = mosaico_escuintla.normalizedDifference(['B8', 'B4']).rename('NDVI')

# 5. Análisis estadístico regional
# Extraemos el valor promedio en un área de influencia (buffer) de 1 km
valor_escuintla = ndvi_escuintla.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=punto_escuintla.buffer(1000),
    scale=10 # Resolución nativa de las bandas Sentinel-2 (10m)
).get('NDVI')

# Salida de resultados
print(f"✅ NDVI promedio en San Juan Panamá (Enero 2024): {valor_escuintla.getInfo()}")

En esta fase, comparamos el estado actual de la vegetación frente a la línea base de 2024. El objetivo es identificar variaciones significativas en el vigor fotosintético que puedan indicar procesos de deforestación, degradación o recuperación del ecosistema.

In [ ]:
# 1. Obtención de datos Sentinel-2 para el periodo actual (Enero 2026)
# El filtrado se mantiene idéntico para asegurar una comparación estandarizada.
coleccion_2026 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(punto_escuintla)
    .filterDate('2026-01-01', '2026-01-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

# 2. Procesamiento de la imagen compuesta y cálculo de NDVI
mosaico_2026 = coleccion_2026.median()
ndvi_2026 = mosaico_2026.normalizedDifference(['B8', 'B4']).rename('NDVI_2026')

# 3. Extracción de estadística espacial (Media regional)
valor_2026 = ndvi_2026.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=punto_escuintla.buffer(1000),
    scale=10
).get('NDVI_2026')

# 4. Análisis comparativo (Cálculo del Delta de NDVI)
# Definimos la línea base de 2024 para calcular la variación neta.
ndvi_24 = 0.6761738769606547
ndvi_26 = valor_2026.getInfo()
diferencia = ndvi_26 - ndvi_24

# --- REPORTE DE RESULTADOS ---
print(f"📊 Valor base NDVI (2024): {ndvi_24}")
print(f"📊 Valor actual NDVI (2026): {ndvi_26}")
print(f"📉 Variación neta (Delta): {diferencia}")

# 5. Lógica de clasificación de cambios
# Se establecen umbrales de ±0.1 para filtrar ruido y detectar cambios reales.
if diferencia < -0.1:
    print("⚠️ ALERTA: Pérdida significativa de cobertura detectada.")
elif diferencia > 0.1:
    print("🌱 CRECIMIENTO: Incremento en la densidad de vegetación detectado.")
else:
    print("✅ ESTABILIDAD: Sin cambios significativos en el área de estudio.")

Visualización Interactiva Cartográfica
Para validar espacialmente los resultados, generamos un mapa interactivo utilizando la librería geemap. Se renderiza la capa de NDVI de 2026 con una paleta de colores que resalta el vigor vegetativo: el rojo indica ausencia de vegetación o suelo desnudo, mientras que el verde intenso representa coberturas forestales densas y saludables.

In [ ]:
import geemap


# 1. Inicialización del Mapa Interactivo
# Centramos la vista en las coordenadas de San Juan Panamá, Escuintla, con un zoom óptico.
Map = geemap.Map(center=[15.29, -92.60], zoom=13)

# 2. Configuración de Parámetros de Visualización para NDVI
# Definimos el rango dinámico (0 a 1) y la rampa de color (Rojo-Amarillo-Verde).
# Rojo = Bajo vigor; Verde = Alto vigor.
parametros_vis = {
    'min': 0,
    'max': 1,
    'palette': ['#e74c3c', '#f1c40f', '#2ecc71'] # Usamos códigos HEX para colores más profesionales
}

# 3. Incorporación de Capas Ráster y Vectoriales
# Añadimos el ráster de NDVI calculado para 2026 y el punto de referencia.
Map.addLayer(ndvi_2026, parametros_vis, 'Salud Vegetal (Enero 2026)')
Map.addLayer(punto_escuintla, {'color': '#3498db'}, 'Ubicación: San Juan Panamá')

# 4. Despliegue del Mapa en el Notebook
Map

Exportación de Resultados a Google Drive
Para asegurar la persistencia de los datos y permitir análisis posteriores en software SIG externo (como QGIS), exportamos el ráster de NDVI procesado. Se define una región rectangular que abarca el área de estudio y se utiliza el formato Cloud Optimized GeoTIFF para optimizar el rendimiento del archivo.

In [ ]:
# 1. Definición del área de exportación
# Generamos un área rectangular (bounding box) de 2km x 2km para el archivo de salida.
geometria_exportar = punto_escuintla.buffer(2000).bounds()

# 2. Configuración de la tarea de exportación a Google Drive
tarea = ee.batch.Export.image.toDrive(
    image=ndvi_2026,
    description='NDVI_SanJuanPanama_2026',
    folder='Proyecto_Monitoreo_Chiapas',
    scale=10,                      # Resolución espacial de Sentinel-2
    region=geometria_exportar,
    fileFormat='GeoTIFF',
    formatOptions={'cloudOptimized': True}
)

# 3. Ejecución y monitoreo de la tarea
# La tarea se procesa en los servidores de Google Earth Engine de forma asíncrona.
print("🚀 Iniciando exportación a Google Drive...")
tarea.start()

import time
while tarea.active():
    print(f"⌛ Estado de la tarea: {tarea.status()['state']}...", end="\r")
    time.sleep(15)

print(f"\n✅ Proceso completado. Archivo disponible en la carpeta 'Proyecto_Monitoreo_Chiapas'.")